# Brain tumor MRI: fine-tuning experiments (Colab)

## RUN ON A COLAB GPU
**Runtime > Change runtime type > GPU (T4 or better).** Training on CPU is impractically slow.

**NOT YET RUN: no results are claimed.** This notebook only calls the scripts in `ml/`. Educational project, not a medical device. The dataset (Kaggle, Masoud Nickparvar, 'Brain Tumor MRI Dataset') is not in the repository; check the dataset page for its current license.

## 1. Get the code and install
Replace `<REPO_URL>` with the project repository URL, or upload the `ml/` folder.

In [ ]:
!git clone <REPO_URL> repo
%cd repo
!pip install -q -r ml/requirements-ml.txt

## 2. Put the dataset in place
Upload your own copy so that `dataset/Training/<class>/` and `dataset/Testing/<class>/` exist (classes: glioma, meningioma, notumor, pituitary). With Drive: mount it and set `BRAIN_DATA_DIR`.

In [ ]:
import os
# from google.colab import drive; drive.mount('/content/drive')
# os.environ['BRAIN_DATA_DIR'] = '/content/drive/MyDrive/dataset'
print(os.environ.get('BRAIN_DATA_DIR', 'using ./dataset'))

## 3. Duplicate audit and duplicate-aware split

In [ ]:
!python ml/data_audit.py

## 4. Train (experiments E1-E6; see ml/EXPERIMENT_LOG.md)
Each run saves a checkpoint under `ml/outputs/checkpoints/`. Repeat with the seeds in `ml/config.py`.

In [ ]:
# E1 head-only baseline (frozen backbone)
!python ml/train_finetune.py --model resnet50 --seed 42 --skip-stage2

In [ ]:
# E2 two-stage, unfreeze conv5
!python ml/train_finetune.py --model resnet50 --unfreeze conv5 --seed 42

In [ ]:
# E3 two-stage, unfreeze conv4 + conv5
!python ml/train_finetune.py --model resnet50 --unfreeze conv4 --seed 42

In [ ]:
# E5 EfficientNetV2-S (optional)
!python ml/train_finetune.py --model efficientnetv2s --seed 42

## 5. Evaluate on the test set
The test set is used only here. Adjust `--model` to the checkpoint you trained.

In [ ]:
!python ml/evaluate.py --model ml/outputs/checkpoints/resnet50_conv5_seed42.keras --name resnet50_conv5_seed42

## 6. Look at the outputs
Copy numbers into `ml/EXPERIMENT_LOG.md` by hand from `metrics.json`. Do not report anything that a completed run did not produce.

In [ ]:
import json
print(json.dumps(json.load(open('ml/outputs/eval/resnet50_conv5_seed42/metrics.json')), indent=2))

In [ ]:
from IPython.display import Image, display
for f in ('gradcam_correct.png','gradcam_wrong.png'):
    display(Image('ml/outputs/eval/resnet50_conv5_seed42/'+f))